In [9]:
import random
import heapq
from collections import deque

In [10]:
def generate_maze(size=11, seed=42):
    if size % 2 == 0:
        size += 1
    random.seed(seed)
    maze = [[1]*size for _ in range(size)]

    def carve(x, y):
        dirs = [(2,0), (-2,0), (0,2), (0,-2)]
        random.shuffle(dirs)
        for dx, dy in dirs:
            nx, ny = x+dx, y+dy
            if 0 < nx < size-1 and 0 < ny < size-1 and maze[nx][ny] == 1:
                maze[nx][ny] = 0
                maze[x+dx//2][y+dy//2] = 0
                carve(nx, ny)

    maze[1][1] = 0
    carve(1,1)

    maze[0][1] = 0
    maze[0][0] = 'S'
    maze[size-1][size-2] = 0
    maze[size-1][size-1] = 'G'

    # Add cycles by removing some random walls
    for _ in range(size):
        rx, ry = random.randint(1, size-2), random.randint(1, size-2)
        if maze[rx][ry] == 1:
            maze[rx][ry] = 0

    return maze

In [11]:
def find_points(maze):
    start = goal = None
    for i in range(len(maze)):
        for j in range(len(maze[0])):
            if maze[i][j] == 'S':
                start = (i, j)
            if maze[i][j] == 'G':
                goal = (i, j)
    return start, goal


def get_neighbors(pos, maze):
    x, y = pos
    directions = [(1,0), (-1,0), (0,1), (0,-1)]

    neighbors = []
    for dx, dy in directions:
        nx, ny = x+dx, y+dy
        if 0 <= nx < len(maze) and 0 <= ny < len(maze[0]):
            if maze[nx][ny] != 1:
                neighbors.append((nx, ny))
    return neighbors

In [12]:
def bfs(maze):
    start, goal = find_points(maze)
    queue = deque([(start, [start])])
    visited = set([start])

    while queue:
        current, path = queue.popleft()
        if current == goal:
            return path

        for neighbor in get_neighbors(current, maze):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None

In [13]:
def dfs(maze):
    start, goal = find_points(maze)
    stack = [(start, [start])]
    visited = set()

    while stack:
        current, path = stack.pop()

        if current == goal:
            return path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in get_neighbors(current, maze):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return None

In [14]:
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(maze):
    start, goal = find_points(maze)
    heap = [(0, start, [start])]
    visited = set()

    while heap:
        cost, current, path = heapq.heappop(heap)

        if current == goal:
            return path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in get_neighbors(current, maze):
            new_cost = len(path)
            priority = new_cost + heuristic(neighbor, goal)
            heapq.heappush(heap, (priority, neighbor, path + [neighbor]))

    return None

In [15]:
def print_maze(maze):
    for row in maze:
        print(" ".join(str(x) for x in row))


def draw_all_paths(maze, bfs_path, dfs_path, astar_path):
    maze_copy = [row[:] for row in maze]

    # BFS
    for (x, y) in bfs_path:
        if maze_copy[x][y] not in ['S', 'G']:
            maze_copy[x][y] = 'B'

    # DFS
    for (x, y) in dfs_path:
        if maze_copy[x][y] == 0:
            maze_copy[x][y] = 'D'

    # A*
    for (x, y) in astar_path:
        if maze_copy[x][y] == 0:
            maze_copy[x][y] = 'A'

    return maze_copy

In [16]:
maze = generate_maze(size=11, seed=0)

bfs_path = bfs(maze)
dfs_path = dfs(maze)
astar_path = astar(maze)

print("Original Maze:\n")
print_maze(maze)

print("\nCombined Paths (BFS=B, DFS=D, A*=A):\n")
combined = draw_all_paths(maze, bfs_path, dfs_path, astar_path)
print_maze(combined)

print("\nLengths:")
print("BFS:", len(bfs_path))
print("DFS:", len(dfs_path))
print("A* :", len(astar_path))

Original Maze:

S 0 1 1 1 1 1 1 1 1 1
1 0 0 0 1 0 0 0 0 0 1
1 1 0 0 1 0 1 0 1 0 1
1 0 1 0 1 0 1 0 1 0 1
1 0 1 0 1 1 1 0 1 0 1
1 0 1 0 1 0 0 0 0 0 1
1 0 1 0 1 0 1 1 1 0 1
1 0 1 0 0 0 1 0 0 0 1
1 0 1 1 1 1 0 0 1 1 1
1 0 0 0 0 0 0 0 0 0 1
1 1 1 1 1 1 1 1 1 0 G

Combined Paths (BFS=B, DFS=D, A*=A):

S B 1 1 1 1 1 1 1 1 1
1 B B D 1 0 0 0 0 0 1
1 1 B B 1 0 1 0 1 0 1
1 0 1 B 1 0 1 0 1 0 1
1 0 1 B 1 1 1 0 1 0 1
1 0 1 B 1 B B B B B 1
1 0 1 B 1 B 1 1 1 B 1
1 0 1 B B B 1 B B B 1
1 0 1 1 1 1 D B 1 1 1
1 0 0 0 0 0 D B B B 1
1 1 1 1 1 1 1 1 1 B G

Lengths:
BFS: 29
DFS: 31
A* : 29
